In [1]:
import nmrglue as ng
import numpy as np
from scipy.signal import find_peaks
from scipy.optimize import curve_fit


In [2]:

# Load H8 processed spectrum
dic, data = ng.bruker.read_pdata("data/400MHz/H8/10/pdata/1")
udic = ng.bruker.guess_udic(dic, data)
uc = ng.fileiobase.uc_from_udic(udic)
ppm = uc.ppm_scale()

# Focus on TSP reference peak near 0.0 ppm
mask = (ppm > -0.5) & (ppm < 0.5)
ppm_region = ppm[mask]
data_region = np.real(data[mask])

# Find peak maximum
peak_idx = np.argmax(data_region)
half_max = data_region[peak_idx] / 2
above_half = data_region > half_max
fwhm_ppm = np.sum(above_half) * abs(ppm_region[1] - ppm_region[0])
fwhm_hz = fwhm_ppm * 400.11

print(f"FWHM = {fwhm_ppm:.4f} ppm = {fwhm_hz:.2f} Hz")
print(f"Bin size = 0.04 ppm = {0.04 * 400.11:.2f} Hz")
print(f"Ratio linewidth/bin = {fwhm_hz / (0.04 * 400.11):.2f}")

FWHM = 0.0041 ppm = 1.63 Hz
Bin size = 0.04 ppm = 16.00 Hz
Ratio linewidth/bin = 0.10


Classification starts degrading when LB broadening approaches the bin size of 16 Hz. This explains exactly why your 400 MHz model collapses at LB = 10 Hz — you are approaching the point where linewidth equals bin size and peaks start merging across bins.
